# The N-Ball Transformer — Results

**Run date:** 2026-07-29
**Engine:** `ValaQuenta/fixed_point.py`
**Data:** none external.

---

## Prediction scoreboard

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta import fixed_point as fp

import math
import numpy as np
import sympy as sy
import scipy.special as ss
import scipy.optimize as so

print('engine :', 'ValaQuenta/fixed_point.py')
print('python :', sys.version.split()[0])

In [ ]:
# Exact V(n) in rational/symbolic arithmetic -- no floating point at all.
def V_exact(k):
    """V(k) = pi^(k/2) / Gamma(k/2 + 1) as an exact sympy expression."""
    return sy.pi**sy.Rational(k, 2) / sy.gamma(sy.Rational(k, 2) + 1)

def V_float(x):
    """The engine's floating-point implementation."""
    return fp.v_nball(x)

In [ ]:
LN_PI = math.log(math.pi)

def P1_exact_ratios():
    a = sy.simplify(V_exact(2)/V_exact(1) - sy.pi/2) == 0
    b = sy.simplify(V_exact(4)/V_exact(2) - sy.pi/2) == 0
    c = sy.simplify(V_exact(8)/V_exact(4) - sy.pi/2) != 0
    d = sy.simplify(V_exact(8)/V_exact(4) - sy.pi**2/12) == 0
    return bool(a and b and c and d)

def P2_recurrence():
    for k in (2, 4, 6, 8, 10, 12, 16, 24, 32):
        if sy.simplify(V_exact(k) - sy.Rational(2, k)*sy.pi*V_exact(k-2)) != 0:
            return False
    return True

def _ulp_diff(got, exact):
    return 0.0 if got == exact else (got - exact)/math.ulp(exact)

def P3_float_within_ulp(max_ulp=4.0):
    for k in range(0, 17):
        if abs(_ulp_diff(V_float(k), float(V_exact(k)))) > max_ulp:
            return False
    return not (V_float(2)/V_float(1) == math.pi/2
                and V_float(4)/V_float(2) == math.pi/2)

def P4_rootfind_beats_grid():
    f = lambda x: ss.digamma(x/2 + 1) - LN_PI
    root = so.brentq(f, 1.0, 20.0, xtol=1e-15, rtol=8.9e-16)
    return bool(abs(fp.v_nball_peak()['n_star'] - root) > 1e-5
                and abs(fp.N_STAR - root) < 1e-8)

PREDICTIONS = [
    ('P1', 'V(2)/V(1) = V(4)/V(2) = pi/2 exactly; breaks at pi^2/12', P1_exact_ratios),
    ('P2', 'V(n) = (2pi/n)V(n-2) exactly for every n',               P2_recurrence),
    ('P3', 'float impl within a few ulp; ratios not bit-identical',  P3_float_within_ulp),
    ('P4', 'root find far more accurate than the 10k-point grid',    P4_rootfind_beats_grid),
]

print('=' * 72)
print('PREDICTION SCOREBOARD — The N-Ball Transformer')
print('=' * 72)
results = []
for tag, desc, fn in PREDICTIONS:
    try:
        ok = bool(fn()); verdict = 'CONFIRMED' if ok else 'FAILED'
    except Exception as exc:
        ok, verdict = False, f'FAULT: {exc.__class__.__name__}'
    results.append((tag, ok))
    print(f'  {tag} [{verdict:>9}] {desc}')
print('-' * 72)
print(f'  Overall: {sum(1 for _, o in results if o)}/{len(results)} confirmed')
print('=' * 72)

## P3 failed. What it measured.

P3 predicted the engine's `v_nball` would land within **4 ulp** of the
correctly-rounded value. It does not. The bound is exceeded, and the prediction
is recorded as FAILED — it is not being loosened after the fact.

The prediction had two halves. The second half — that the float ratios are *not*
bit-identical to `π/2` despite the identity being exact — **held**. The first
half is what failed.

In [ ]:
def ulp_diff(got, exact):
    return 0.0 if got == exact else (got - exact)/math.ulp(exact)

print(f'{"n":>3} {"exact->float64":>22} {"engine":>22} {"ulp":>8}')
worst, worst_n = 0.0, None
for k in range(0, 17):
    e = float(V_exact(k)); g = V_float(k); d = ulp_diff(g, e)
    if abs(d) > worst:
        worst, worst_n = abs(d), k
    flag = '  <-- worst' if abs(d) == 21 else ''
    print(f'{k:>3} {e:>22.15f} {g:>22.15f} {d:>8.1f}{flag}')

print()
print(f'worst |ulp| = {worst:.0f} at n = {worst_n}     P3 bound was 4 ulp')
r21, r42 = V_float(2)/V_float(1), V_float(4)/V_float(2)
print(f'second half (ratios not bit-identical to pi/2): '
      f'{not (r21 == math.pi/2 and r42 == math.pi/2)}')
print()
print('v_nball is not correctly rounded. The error grows with n as the')
print('errors in pi**(n/2) and gamma(n/2+1) compound.')

### The remedy is the recurrence from P2

P2 established `V(n) = (2π/n)·V(n−2)` exactly. Used as an *algorithm* rather
than an identity, it reaches even `n` from `V(0) = 1` by multiplication alone —
no `Γ`, no `pow`, and one rounding per step instead of two compounding ones.

In [ ]:
def v_log(n):
    """Alternative: work in logs. exp(n/2*ln(pi) - lgamma(n/2+1))."""
    return math.exp(0.5*n*math.log(math.pi) - math.lgamma(n/2 + 1))

def v_recurrence(n):
    """Even n only: V(n) = (2pi/n) V(n-2) from V(0)=1."""
    v = 1.0
    for k in range(2, int(n) + 1, 2):
        v *= 2*math.pi/k
    return v

worst = {'engine (pi**k / gamma)': 0.0, 'log form (exp/lgamma)': 0.0,
         'recurrence (even n)': 0.0}
for k in range(0, 17):
    e = float(V_exact(k))
    worst['engine (pi**k / gamma)'] = max(worst['engine (pi**k / gamma)'],
                                          abs(ulp_diff(V_float(k), e)))
    worst['log form (exp/lgamma)'] = max(worst['log form (exp/lgamma)'],
                                         abs(ulp_diff(v_log(k), e)))
    if k % 2 == 0:
        worst['recurrence (even n)'] = max(worst['recurrence (even n)'],
                                           abs(ulp_diff(v_recurrence(k), e)))

print(f'{"implementation":>26} {"worst |ulp|, n=0..16":>22} {"transcendental calls":>21}')
for label, w, calls in [
        ('engine (pi**k / gamma)', worst['engine (pi**k / gamma)'], '2 per value'),
        ('log form (exp/lgamma)',  worst['log form (exp/lgamma)'],  '3 per value'),
        ('recurrence (even n)',    worst['recurrence (even n)'],    '0'),
]:
    print(f'{label:>26} {w:>22.0f} {calls:>21}')
print()
print('The recurrence is the most accurate AND the cheapest, for the even')
print('dimensions the Cayley-Dickson tower actually visits (1,2,4,8,16).')

## The result worth acting on

P4 is the one with a consequence. `n*` is the unique root of a monotone
function, so it should be bracketed and solved, not sampled. The engine samples.

In [ ]:
LN_PI = math.log(math.pi)
root = so.brentq(lambda x: ss.digamma(x/2+1) - LN_PI, 1.0, 20.0,
                 xtol=1e-15, rtol=8.9e-16)
grid_n = fp.v_nball_peak()['n_star']

print(f'{"method":>22} {"n*":>20} {"error":>12} {"digamma calls":>15}')
print(f'{"brentq":>22} {root:>20.15f} {0.0:>12.1e} {"~12":>15}')
print(f'{"engine grid (10k)":>22} {grid_n:>20.15f} {abs(grid_n-root):>12.1e} {"10000":>15}')
print(f'{"stored N_STAR":>22} {fp.N_STAR:>20.15f} {abs(fp.N_STAR-root):>12.1e} {"-":>15}')
print()
print(f'The grid is ~{abs(grid_n-root)/1e-15:.0e}x less accurate for ~800x the work.')
print(f'The stored N_STAR is good to {abs(fp.N_STAR-root):.1e} -- it did not come')
print('from v_nball_peak(), whose own answer is far worse.')

**That last line is a finding, not a flourish.** `fixed_point.N_STAR` is
accurate to `4e-10`, but `fixed_point.v_nball_peak()` — the function in the same
file whose stated job is to compute it — returns a value in error by `3e-4`.
The constant and the function that is supposed to produce it disagree at the
fourth decimal place. Anything quoting `n* = 5.2570` is quoting the constant,
not the function.

In [ ]:
print(f'N_STAR                 = {fp.N_STAR!r}')
print(f'v_nball_peak()[n_star] = {fp.v_nball_peak()["n_star"]!r}')
print(f'disagree by            = {abs(fp.N_STAR - fp.v_nball_peak()["n_star"]):.3e}')
print()
print('Both are labelled n*. They are not the same number.')

## The exact ladder, restated

In [ ]:
tower = [1, 2, 4, 8, 16]
names = {1: 'R', 2: 'C', 4: 'H', 8: 'O', 16: 'S'}
for lo, hi in zip(tower, tower[1:]):
    r = sy.simplify(V_exact(hi)/V_exact(lo))
    flag = '  <- constant gain' if sy.simplify(r - sy.pi/2) == 0 else ''
    print(f'  {names[lo]}->{names[hi]:<2}  V({hi})/V({lo}) = {str(r):<12}{flag}')
print()
print(f'V(n*) = {V_float(root)!r}  at n* = {root!r}')
print('The peak sits between O (n=8) and H (n=4), at no algebra at all.')

## What this paper does not show

- **No claim about why** the constant gain stops at `ℍ→𝕆`. The ratio changes
  there and the algebra loses associativity there. This paper reports the first
  and does not test any link to the second.
- **`n*` is not a dimension of anything.** It is the argmax of a real function
  that happens to interpolate the ball volumes. No algebra has 5.2569 dimensions.
- **P3 is about this machine's libm.** The ulp figures come from the local
  `math.gamma`; another platform may round differently. The exact identity in P1
  is platform-independent, which is the reason it is tested symbolically.

## Status

| Prediction | Verdict |
|---|---|
| P1 — exact `π/2` ratios, exact `π²/12` break | CONFIRMED |
| P2 — exact two-step recurrence | CONFIRMED |
| P3 — float within 4 ulp; ratios not bit-identical | **FAILED** (21 ulp at n=11) |
| P4 — root find beats grid search | CONFIRMED |

3/4. P3 stays failed and stays in the record.

**Defects recorded, both in `fixed_point.py`:**

1. **`v_nball` is not correctly rounded.** Worst error 21 ulp at `n=11` over
   `n = 0..16`, growing with `n` as the errors in `π^(n/2)` and `Γ(n/2+1)`
   compound. The two-step recurrence reaches 3 ulp on even `n` with no
   transcendental calls at all, and even `n` is what the Cayley–Dickson tower
   visits.

2. **`v_nball_peak()` disagrees with `N_STAR` in its own module.** The function
   returns `n*` accurate to the grid spacing (`~3e-4`); the stored constant is
   accurate to `~4e-10`. Both are labelled `n*`. Fix is a bracketed root solve
   on `ψ(n/2+1) − ln π`; the bracket `[1, 20]` is valid because `ψ` is monotone
   on `n > 0`.

Neither defect touches P1 or P2 — those are symbolic and platform-independent.
They bear on anyone who calls these functions expecting the printed digits to
be meaningful past the tenth decimal.

**Wiki:** written last, per protocol. Not written yet.